In [5]:
from pyspark.sql import SparkSession

In [6]:
spark= SparkSession.builder.appName("casestudyApp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 04:17:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
products=spark.read.csv("casestudy/data/products.csv",header=True)
returns=spark.read.csv("casestudy/data/returns.csv",header=True)
orders=spark.read.csv("casestudy/data/orders.csv",header=True)
Items=spark.read.csv("casestudy/data/order_items.csv",header=True)
customers=spark.read.csv("casestudy/data/customers.csv",header=True)

26/06/16 04:17:13 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [12]:
#q1
total_customers = customers.count()
dis_total_customers= customers.distinct().count()
print(f"Total number of customers: {total_customers}")
print(f"Total number of distinct customers: {dis_total_customers}")

[Stage 38:=============================>                            (1 + 1) / 2]

Total number of customers: 100000
Total number of distinct customers: 100000


In [13]:
total_products=products.distinct().count()
print(f"Total number of distinct products: {total_products}")

print(f"Total number of orders: {orders.count()}")
print(f"Total number of returned orders: {returns.count()}")

Total number of distinct products: 50000


Total number of orders: 1000000
Total number of returned orders: 100000


In [9]:
# q2
print(order_items.product_id,sum(selling_price) as total_sales group by product_id)

SyntaxError: invalid syntax (2093401944.py, line 2)

In [15]:
from pyspark.sql import functions as F

# 1. Group by product_id and calculate total sales
sales_df = Items.groupBy("product_id").agg(F.sum("selling_price").alias("total_sales"))
# 2. Display the result
sales_df.show()

[Stage 56:=============================>                            (1 + 1) / 2]

+----------+------------------+
|product_id|       total_sales|
+----------+------------------+
|     17401|36111.049999999996|
|     16576|13884.550000000001|
|     26112|           38256.5|
|     40726|           5923.67|
|     26439|          40416.62|
|     47940| 61250.05999999998|
|     30002| 42750.21000000001|
|     36470|49641.869999999995|
|     49630|21877.219999999994|
|      3959|           4861.12|
|     32812|          38364.48|
|      7762| 9870.360000000002|
|     47092|3233.3500000000004|
|     15269| 65330.23000000001|
|     35444| 46534.17999999999|
|     39659|           2762.87|
|     44423|          17981.59|
|      4937|          57927.61|
|     44446|21938.339999999997|
|      9009|          65414.95|
+----------+------------------+
only showing top 20 rows



In [16]:
# 1. Register the DataFrame as a temporary SQL table
Items.createOrReplaceTempView("order_items_table")

# 2. Write your standard SQL query
sql_query = """
    SELECT product_id, SUM(selling_price) AS total_sales 
    FROM order_items_table 
    GROUP BY product_id
"""

# 3. Run the query and show the results
sales_sql_df = spark.sql(sql_query)
sales_sql_df.show()

[Stage 59:=============================>                            (1 + 1) / 2]

+----------+------------------+
|product_id|       total_sales|
+----------+------------------+
|     17401|36111.049999999996|
|     16576|13884.550000000001|
|     26112|           38256.5|
|     40726|           5923.67|
|     26439|          40416.62|
|     47940| 61250.05999999998|
|     30002| 42750.21000000001|
|     36470|49641.869999999995|
|     49630|21877.219999999994|
|      3959|           4861.12|
|     32812|          38364.48|
|      7762| 9870.360000000002|
|     47092|3233.3500000000004|
|     15269| 65330.23000000001|
|     35444| 46534.17999999999|
|     39659|           2762.87|
|     44423|          17981.59|
|      4937|          57927.61|
|     44446|21938.339999999997|
|      9009|          65414.95|
+----------+------------------+
only showing top 20 rows



In [19]:
#q3
from pyspark.sql import functions as F
top10_df = Items.groupBy("product_id").agg(F.sum("selling_price").alias("total_sales")).orderBy(F.desc("total_sales")).limit(10)
# 2. Display the result
top10_df.show()

[Stage 63:==========================================================(2 + 0) / 2]

+----------+-----------------+
|product_id|      total_sales|
+----------+-----------------+
|      7025|        101171.16|
|     15951|98236.37000000001|
|     35314|97258.20999999996|
|     31322|         96562.18|
|     28316|96550.32999999999|
|      4967|94977.50999999998|
|     28311|94770.93999999999|
|     30359|94715.01999999999|
|     44886|94187.85999999999|
|     42099|93987.76000000004|
+----------+-----------------+



In [20]:
from pyspark.sql import functions as F
top10_df = Items.join(products, on="product_id", how="inner").groupBy("product_id","product_name").agg(F.sum("selling_price").alias("total_sales")).orderBy(F.desc("total_sales")).limit(10)
# 2. Display the result
top10_df.show()   #* why multiply??

[Stage 67:>                                                         (0 + 2) / 2]

+----------+-------------+-----------------+
|product_id| product_name|      total_sales|
+----------+-------------+-----------------+
|      7025| Product_7025|        101171.16|
|     15951|Product_15951|98236.37000000001|
|     35314|Product_35314|97258.20999999996|
|     31322|Product_31322|         96562.18|
|     28316|Product_28316|96550.32999999999|
|      4967| Product_4967|94977.50999999998|
|     28311|Product_28311|94770.93999999999|
|     30359|Product_30359|94715.01999999999|
|     44886|Product_44886|94187.85999999999|
|     42099|Product_42099|93987.76000000004|
+----------+-------------+-----------------+



In [24]:
# 1. Register the DataFrame as a temporary SQL table
Items.createOrReplaceTempView("order_items_table")
products.createOrReplaceTempView("products_table")

# 2. Write your standard SQL query
sql_query = """
    SELECT i.product_id,p.product_name, SUM(i.selling_price) AS total_sales 
    FROM order_items_table i inner join products_table p
    ON i.product_id = p.product_id
    GROUP BY i.product_id, p.product_name
    ORDER BY total_sales desc
    LIMIT 10
"""

# 3. Run the query and show the results
sales_sql_df = spark.sql(sql_query)
sales_sql_df.show()

[Stage 71:=============================>                            (1 + 1) / 2]

+----------+-------------+-----------------+
|product_id| product_name|      total_sales|
+----------+-------------+-----------------+
|      7025| Product_7025|        101171.16|
|     15951|Product_15951|98236.37000000001|
|     35314|Product_35314|97258.20999999996|
|     31322|Product_31322|         96562.18|
|     28316|Product_28316|96550.32999999999|
|      4967| Product_4967|94977.50999999998|
|     28311|Product_28311|94770.93999999999|
|     30359|Product_30359|94715.01999999999|
|     44886|Product_44886|94187.85999999999|
|     42099|Product_42099|93987.76000000004|
+----------+-------------+-----------------+



In [9]:
#q4:Calculate monthly sales trends for the last available year.
Items.createOrReplaceTempView("order_items_table")
orders.createOrReplaceTempView("orders_table")

query="""
      select sum(i.selling_price) as sales,month(o.order_date) as order_month
      from order_items_table i inner join orders_table o
      on i.order_id=o.order_id
      where year(o.order_date) in(select max(year(order_date)) from orders_table)
      group by month(o.order_date)
      order by order_month asc
      """

sales_sql_df = spark.sql(query)
sales_sql_df.show()

[Stage 12:=============================>                            (1 + 1) / 2]

+--------------------+-----------+
|               sales|order_month|
+--------------------+-----------+
|1.4807304093999895E8|          1|
|1.3861922224000004E8|          2|
|1.4762569981000108E8|          3|
| 1.425444345899993E8|          4|
| 1.481984605599982E8|          5|
|1.4395244331000018E8|          6|
|1.4785272724999964E8|          7|
|1.4692396998000103E8|          8|
|1.4389496400000024E8|          9|
| 1.472589345899999E8|         10|
| 1.445234103299995E8|         11|
| 1.478689167300004E8|         12|
+--------------------+-----------+



In [22]:
from pyspark.sql import functions as F
# res=Items.join(orders, on="order_id", how="inner").F.month(order_date).alias("order_month").agg(F.sum("selling_price").alias("total_sales")).groupBy(order_month).orderBy(F.desc("total_sales"))
# # 2. Display the result
# res.show() 

res=Items.join(orders, on="order_id", how="inner")\
         .withColumn("order_month", F.month("order_date"))\
         .where("year(order_date)=(select max(year(order_date)) from orders_table)")\
         .groupBy("order_month")\
         .agg(F.sum("selling_price")\
         .alias("total_sales"))\
         .orderBy(F.desc("order_month"))
res.show()      # here we use both sql and df

[Stage 45:=============================>                            (1 + 1) / 2]

+-----------+--------------------+
|order_month|         total_sales|
+-----------+--------------------+
|         12| 1.478689167300004E8|
|         11| 1.445234103299995E8|
|         10| 1.472589345899999E8|
|          9|1.4389496400000024E8|
|          8|1.4692396998000103E8|
|          7|1.4785272724999964E8|
|          6|1.4395244331000018E8|
|          5| 1.481984605599982E8|
|          4| 1.425444345899993E8|
|          3|1.4762569981000108E8|
|          2|1.3861922224000004E8|
|          1|1.4807304093999895E8|
+-----------+--------------------+



In [ ]:
#q5: find the return percentage for each product category
# ret%, p_id,p_name
Items.createOrReplaceTempView("order_items")
returns.createOrReplaceTempView("returns")
products.createOrReplaceTempView("products")

query= """
       select (count(r.return_id)/count(i.order_item_id))*100
       as return_percentage,
       i.product_id,p.product_name
       FROM order_items i
    INNER JOIN products p ON i.product_id = p.product_id
    LEFT JOIN returns r ON i.order_id = r.order_id
       group by i.product_id, p.product_name
       """
sales_df=spark.sql(query)
sales_df.show()